# Product listings — modular version
Refactored from `legacy/bad_practice.ipynb`; the original is preserved.
Loading, validation, image processing, prompt construction, API access, response
parsing, result formatting, and saving each have separate functions.
The two workflow functions only coordinate those steps.

Run the definition cells and offline checks first. The final example is disabled
by default because it downloads data, calls the paid API, and writes output files.
Install `datasets`, `pandas`, `Pillow`, and `openai` in your notebook environment.

In [1]:
import base64
import csv
import json
import math
import time
from io import BytesIO
from pathlib import Path
from PIL import Image

REQUIRED_KEYS = {"title", "description", "features", "keywords"}
LISTING_COLUMNS = ["id", "product_name", "category", "title", "description", "features", "keywords"]
ERROR_COLUMNS = ["id", "product_name", "error"]

## Actionable errors
Each protected function reports **Function | Error type and message | Context and source line | Suggestion**.
JSON failures also include the JSON line and column. Original exceptions are chained with
`raise ... from error`, preserving the traceback. The shared decorator keeps error reporting
separate from each function's primary task. Batch processing records the full message in
`errors.csv` and continues; loading and saving failures raise immediately.

Validation currently uses explicit checks, not Pydantic models. Those failures receive
the same diagnostics; Pydantic ValidationError is also supported if models are added later.
Contexts exclude API clients, prompts, response bodies, and encoded images.

In [2]:
from functools import wraps
import inspect
import traceback


class WorkflowError(ValueError):
    """An actionable workflow failure that retains the original exception."""

    def __init__(self, function, original, context, suggestion):
        self.function = function
        self.error_type = type(original).__name__
        self.context = context
        frames = traceback.extract_tb(original.__traceback__)
        origin = next((frame for frame in reversed(frames) if frame.name == function), frames[-1] if frames else None)
        location = f"{origin.filename}:{origin.lineno}" if origin else "unavailable"
        if isinstance(original, json.JSONDecodeError):
            location += f"; JSON line {original.lineno}, column {original.colno}"
        super().__init__(
            f"Function: {function} | Error: {self.error_type}: {original} | "
            f"Context: {context}; source={location} | Suggestion: {suggestion}"
        )


def error_suggestion(error, default):
    """Select a repair suggestion for a specific failure category."""
    if isinstance(error, FileNotFoundError):
        return "Check the path and working directory; create missing parent directories before saving."
    if isinstance(error, PermissionError):
        return "Check folder permissions and whether another application has locked the file."
    if isinstance(error, IsADirectoryError):
        return "Supply a file path rather than a directory."
    if isinstance(error, json.JSONDecodeError):
        return "Fix JSON syntax at the reported line and column; use double quotes and remove trailing commas."
    status = getattr(error, "status_code", None)
    kind = type(error).__name__
    if status == 429 or kind == "RateLimitError":
        return "Check API quota and rate limits; wait before retrying or increase the batch delay."
    if status == 401 or kind == "AuthenticationError":
        return "Check that OPENAI_API_KEY is configured and valid; do not print the key."
    if status == 403:
        return "Check project permissions and access to the selected model."
    if isinstance(error, (ConnectionError, TimeoutError)) or kind in ("APIConnectionError", "APITimeoutError"):
        return "Check your internet connection, proxy, and timeout settings; retry when connectivity returns."
    if status is not None:
        return "Check the API status code, selected model, and request format; retry server errors later."
    if kind == "ValidationError":
        return "Check the validation field locations and supply values matching the expected schema."
    return default


def describe_context(arguments, fields):
    """Describe safe identifiers without including images, credentials, or prompts."""
    parts = [f"{field}={arguments[field]!r}" for field in fields if field in arguments]
    product = arguments.get("product")
    if isinstance(product, dict):
        parts.append(f"product_id={product.get('id')!r}")
        if product.get("image_path"):
            parts.append(f"image_path={product['image_path']!r}")
    return ", ".join(parts) or "in-memory input"


def report_errors(suggestion, fields=()):
    """Attach consistent diagnostics to a function without mixing its responsibilities."""
    def decorate(function):
        signature = inspect.signature(function)

        @wraps(function)
        def wrapped(*args, **kwargs):
            arguments = {}
            try:
                bound = signature.bind(*args, **kwargs)
                bound.apply_defaults()
                arguments = bound.arguments
                return function(*args, **kwargs)
            except WorkflowError:
                # Preserve the innermost function rather than replacing its location.
                raise
            except (OSError, json.JSONDecodeError, ValueError, TypeError, KeyError, AttributeError) as error:
                raise WorkflowError(function.__name__, error, describe_context(arguments, fields),
                                    error_suggestion(error, suggestion)) from error
            except Exception as error:
                # SDK/dataset exceptions also retain their concrete type and cause.
                raise WorkflowError(function.__name__, error, describe_context(arguments, fields),
                                    error_suggestion(error, suggestion)) from error
        return wrapped
    return decorate

## Loading: read data only

In [3]:
@report_errors('Check the dataset name, split, internet access, and installation of datasets.', fields=('dataset_name', 'split'))
def load_product_dataset(dataset_name="ashraq/fashion-product-images-small", split="train[:100]"):
    """Load product records from Hugging Face."""
    from datasets import load_dataset
    return list(load_dataset(dataset_name, split=split))


@report_errors('Check that the file contains UTF-8 JSON product records.', fields=('path',))
def load_local_products(path):
    """Load product records from a JSON file."""
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


@report_errors('Check that the path points to a readable, supported image file.', fields=('path',))
def load_image(path):
    """Load an image fully before closing its file."""
    with Image.open(path) as image:
        return image.copy()

## Validation: check inputs and responses only
Local JSON records use the same field names as the dataset, with `image_path`
instead of `image`. The price remains a configurable example value, as in the original.

In [4]:
@report_errors('Provide id, non-empty productDisplayName and articleType, an image or image_path, and a finite non-negative price.', fields=('price',))
def validate_product(product, price):
    """Reject records missing the fields needed to generate a listing."""
    if not isinstance(product, dict):
        raise ValueError("Each product must be an object")
    for key in ("id", "productDisplayName", "articleType"):
        if key not in product or product[key] is None:
            raise ValueError(f"Missing product field: {key}")
    for key in ("productDisplayName", "articleType"):
        if not isinstance(product[key], str) or not product[key].strip():
            raise ValueError(f"{key} must be non-empty text")
    if not isinstance(product.get("image"), Image.Image) and not product.get("image_path"):
        raise ValueError("Product requires a PIL image or image_path")
    if isinstance(price, bool) or not isinstance(price, (int, float)) or not math.isfinite(price) or price < 0:
        raise ValueError("Price must be a finite, non-negative number")


@report_errors('Check the response fields: title, description, keywords must be text; features must be a list of text.', fields=())
def validate_listing(listing):
    """Check the structure and field types of a parsed listing."""
    if not isinstance(listing, dict):
        raise ValueError("Listing must be a JSON object")
    missing = REQUIRED_KEYS - listing.keys()
    if missing:
        raise ValueError(f"Missing keys in response: {sorted(missing)}")
    for key in ("title", "description", "keywords"):
        if not isinstance(listing[key], str) or not listing[key].strip():
            raise ValueError(f"{key} must be non-empty text")
    if not isinstance(listing["features"], list) or not listing["features"] or not all(
        isinstance(feature, str) and feature.strip() for feature in listing["features"]
    ):
        raise ValueError("features must be a non-empty list of non-empty strings")


@report_errors('Use an integer n >= 0 and a finite delay >= 0.', fields=('n', 'delay'))
def validate_batch_options(n, delay):
    """Check batch size and delay before processing any records."""
    if isinstance(n, bool) or not isinstance(n, int) or n < 0:
        raise ValueError("n must be a non-negative integer")
    if isinstance(delay, bool) or not isinstance(delay, (int, float)) or not math.isfinite(delay) or delay < 0:
        raise ValueError("delay must be finite and non-negative")

## Processing: transform values without network or file access

In [5]:
@report_errors('Supply a valid PIL image with non-zero dimensions.', fields=())
def image_to_jpeg_bytes(image):
    """Serialize a PIL image as JPEG, consistently matching the API MIME type."""
    buffer = BytesIO()
    image.convert("RGB").save(buffer, format="JPEG")
    return buffer.getvalue()


@report_errors('Supply image bytes from image_to_jpeg_bytes.', fields=())
def encode_image_to_base64(image_bytes):
    """Encode image bytes as base64 text."""
    return base64.b64encode(image_bytes).decode("ascii")


@report_errors('Validate the product fields and numeric price before building the prompt.', fields=('price',))
def create_product_listing_prompt(product, price):
    product_name = product["productDisplayName"]
    category = product["articleType"]
    additional_info = f"Colour: {product.get('baseColour', '')}, Gender: {product.get('gender', '')}, Usage: {product.get('usage', '')}"
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt


@report_errors('Supply a text prompt and a base64-encoded image.', fields=())
def build_api_input(prompt, encoded_image):
    """Format the text and image into an API request payload."""
    return [{"role": "user", "content": [
        {"type": "input_text", "text": prompt},
        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{encoded_image}"},
    ]}]


@report_errors('Check that the API returned JSON text rather than empty content or a refusal.', fields=())
def parse_listing(response_text):
    """Parse response JSON without validating or saving it."""
    return json.loads(response_text)


@report_errors('Validate the product and listing before formatting the record.', fields=())
def format_listing_record(product, listing):
    """Combine product identifiers with a generated listing."""
    return {"id": product["id"], "product_name": product["productDisplayName"],
            "category": product["articleType"], **{key: listing[key] for key in REQUIRED_KEYS}}


def format_error_record(product, error):
    """Format a product failure for reporting."""
    record = product if isinstance(product, dict) else {}
    return {"id": record.get("id"), "product_name": record.get("productDisplayName", "Unknown"),
            "error": f"{type(error).__name__}: {error}"}


@report_errors('Check that each record contains features as a list of strings.', fields=())
def format_csv_records(results):
    """Flatten feature lists for CSV without modifying the JSON records."""
    return [{**record, "features": " | ".join(record["features"])} for record in results]

## API: make one request only; the client is supplied by the caller

In [6]:
@report_errors('Check the API client, request format, model access, and service availability.', fields=('model',))
def call_listing_api(client, api_input, model="gpt-4o-mini"):
    """Call the API without parsing, validating, or saving its response."""
    return client.responses.create(
        model=model, input=api_input, text={"format": {"type": "json_object"}}
    )

@report_errors("Set OPENAI_API_KEY in the environment and check the openai installation.")
def create_api_client():
    """Initialize the API client from the environment."""
    from openai import OpenAI
    return OpenAI()


## Workflow: coordinate the focused functions

In [7]:
@report_errors('Check the API response has output_text and the product has valid image data.', fields=('model', 'price'))
def generate_listing(product, client, price=29.99, model="gpt-4o-mini"):
    """Coordinate the steps for generating one validated listing."""
    validate_product(product, price)
    image = product.get("image")
    if not isinstance(image, Image.Image):
        image = load_image(product["image_path"])
    image_bytes = image_to_jpeg_bytes(image)
    encoded = encode_image_to_base64(image_bytes)
    prompt = create_product_listing_prompt(product, price)
    api_input = build_api_input(prompt, encoded)
    response = call_listing_api(client, api_input, model)
    listing = parse_listing(response.output_text)
    validate_listing(listing)
    return listing


@report_errors('Supply a sliceable list of product records and valid batch options.', fields=('n', 'delay'))
def process_products(products, client, n=5, delay=0.5, price=29.99, model="gpt-4o-mini"):
    """Coordinate a batch, retaining successful listings and individual failures."""
    validate_batch_options(n, delay)
    results, errors = [], []
    selected = products[:n]
    for index, product in enumerate(selected):
        try:
            listing = generate_listing(product, client, price, model)
            results.append(format_listing_record(product, listing))
        except Exception as error:
            # A failed record must not prevent the remaining products from running.
            errors.append(format_error_record(product, error))
        if index < len(selected) - 1:
            time.sleep(delay)
    return results, errors

## Saving: write already prepared records only

In [8]:
@report_errors('Check the destination path and available disk space.', fields=('path',))
def create_output_directory(path):
    """Create the output directory."""
    Path(path).mkdir(parents=True, exist_ok=True)


@report_errors('Check disk space and ensure records are JSON serializable.', fields=('path',))
def save_json(records, path):
    """Save records as UTF-8 JSON."""
    with Path(path).open("w", encoding="utf-8") as stream:
        json.dump(records, stream, indent=2, ensure_ascii=False, default=str)


@report_errors('Check disk space and ensure record fields match the CSV columns.', fields=('path',))
def save_csv(records, path, columns):
    """Save prepared rows, including a header when there are no records."""
    with Path(path).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(records)

## Offline checkpoint: no credentials, downloads, or paid calls

In [9]:
from types import SimpleNamespace
from tempfile import TemporaryDirectory

sample_listing = {"title": "Blue shirt", "description": "A blue everyday shirt.",
                  "features": ["Blue", "Short sleeves"], "keywords": "shirt, blue"}
sample_product = {"id": 1, "productDisplayName": "Blue shirt", "articleType": "Shirts",
                  "image": Image.new("RGBA", (8, 8), "blue")}
api_calls = []

def fake_create(**request):
    """Return a deterministic response for the offline checkpoint."""
    api_calls.append(request)
    return SimpleNamespace(output_text=json.dumps(sample_listing))

fake_client = SimpleNamespace(responses=SimpleNamespace(create=fake_create))
results, errors = process_products([{}, sample_product], fake_client, n=2, delay=0)
assert len(results) == 1 and len(errors) == 1
assert len(api_calls) == 1  # Invalid records never reach the API.
assert results[0]["title"] == sample_listing["title"]
image_url = api_calls[0]["input"][0]["content"][1]["image_url"]
with Image.open(BytesIO(base64.b64decode(image_url.split(",", 1)[1]))) as image:
    assert image.format == "JPEG" and image.mode == "RGB"
for invalid in ({}, [], {**sample_listing, "features": "wrong type"}):
    try:
        validate_listing(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid listing accepted")
with TemporaryDirectory() as directory:
    output = Path(directory)
    save_json(results, output / "listings.json")
    save_csv(format_csv_records(results), output / "listings.csv", LISTING_COLUMNS)
    save_csv(errors, output / "errors.csv", ERROR_COLUMNS)
    assert load_local_products(output / "listings.json") == results
    with (output / "listings.csv").open(encoding="utf-8", newline="") as stream:
        assert list(csv.DictReader(stream))[0]["features"] == "Blue | Short sleeves"
    save_csv([], output / "errors.csv", ERROR_COLUMNS)
    with (output / "errors.csv").open(encoding="utf-8", newline="") as stream:
        assert list(csv.DictReader(stream)) == []
    local_image = output / "product.png"
    sample_product["image"].save(local_image)
    local_product = {key: value for key, value in sample_product.items() if key != "image"}
    local_product["image_path"] = str(local_image)
    assert generate_listing(local_product, fake_client) == sample_listing
assert process_products([], fake_client, delay=0) == ([], [])
print("Offline checks passed: validation, image processing, API wiring, batching, local loading, and saving.")

Offline checks passed: validation, image processing, API wiring, batching, local loading, and saving.


## Run with real data (optional)
Set `RUN_LIVE = True` to download the first 100 products and generate five listings.
Set `OPENAI_API_KEY` in your environment first. Output files are replaced on each run,
including empty CSVs, so previous errors cannot linger. For local input, replace the
dataset loader with `load_local_products("your_products.json")`.

In [10]:
RUN_LIVE = False

if RUN_LIVE:
    from openai import OpenAI

    products = load_product_dataset()
    client = create_api_client()
    results, errors = process_products(products, client, n=5)
    csv_records = format_csv_records(results)
    output_dir = Path("output")
    create_output_directory(output_dir)
    save_json(results, output_dir / "listings.json")
    save_csv(csv_records, output_dir / "listings.csv", LISTING_COLUMNS)
    save_csv(errors, output_dir / "errors.csv", ERROR_COLUMNS)
    print(f"Done: {len(results)} succeeded, {len(errors)} failed")
    print(f"Saved to {output_dir.resolve()}")

## Single responsibility checkpoint
- Loading functions only read dataset records, JSON, or image files.
- Validation functions only check product, listing, or batch inputs.
- Processing functions only transform images, prompts, responses, or records.
- `call_listing_api` only makes the API request.
- Saving functions only create the destination or write prepared records.
- `generate_listing` and `process_products` coordinate these steps without implementing them.

## Invalid-input checkpoint
Run these offline checks after the existing checkpoint. API failures and permission errors are simulated deterministically; no paid calls are made. Requires the existing openai and pydantic dependencies.

In [11]:
from unittest.mock import patch

def expect_workflow_error(action, function, error_type, context=None):
    """Verify that a failed action reports its origin and a practical repair."""
    try:
        action()
    except WorkflowError as error:
        message = str(error)
        assert error.function == function, message
        assert error.error_type == error_type, message
        assert error.__cause__ is not None
        assert all(label in message for label in ("Function:", "Error:", "Context:", "source=", "Suggestion:"))
        if context:
            assert context in message, message
        return message
    raise AssertionError(f"Expected an error from {function}")

with TemporaryDirectory() as directory:
    folder = Path(directory)
    example_error = expect_workflow_error(
        lambda: load_local_products(folder / "missing.json"),
        "load_local_products", "FileNotFoundError", "missing.json")
    invalid_json = folder / "invalid.json"
    invalid_json.write_text('{"id": }', encoding="utf-8")
    expect_workflow_error(lambda: load_local_products(invalid_json),
                          "load_local_products", "JSONDecodeError", "JSON line 1")
    corrupt_image = folder / "bad.png"
    corrupt_image.write_text("not an image", encoding="utf-8")
    expect_workflow_error(lambda: load_image(corrupt_image), "load_image", "UnidentifiedImageError", "bad.png")
    expect_workflow_error(lambda: save_csv([], folder / "missing" / "out.csv", ERROR_COLUMNS),
                          "save_csv", "FileNotFoundError", "out.csv")
    with patch.object(Path, "open", side_effect=PermissionError("File is locked")):
        expect_workflow_error(lambda: save_json([], folder / "locked.json"),
                              "save_json", "PermissionError", "locked.json")
    with patch.object(Path, "mkdir", side_effect=PermissionError("Access denied")):
        expect_workflow_error(lambda: create_output_directory(folder / "locked"),
                              "create_output_directory", "PermissionError")

expect_workflow_error(lambda: validate_product({}, 29.99), "validate_product", "ValueError")
expect_workflow_error(lambda: validate_batch_options(-1, 0), "validate_batch_options", "ValueError")
expect_workflow_error(lambda: parse_listing("not JSON"), "parse_listing", "JSONDecodeError")
expect_workflow_error(lambda: validate_listing({}), "validate_listing", "ValueError")
expect_workflow_error(lambda: validate_listing({**sample_listing, "features": "wrong"}), "validate_listing", "ValueError")
expect_workflow_error(lambda: image_to_jpeg_bytes(None), "image_to_jpeg_bytes", "AttributeError")
expect_workflow_error(lambda: generate_listing(sample_product, SimpleNamespace(
    responses=SimpleNamespace(create=lambda **kwargs: SimpleNamespace()))),
    "generate_listing", "AttributeError", "product_id=1")

# Use real SDK exception types with a fake transport: no network requests.
import httpx
from openai import APIConnectionError, APITimeoutError, RateLimitError, AuthenticationError, InternalServerError
request = httpx.Request("POST", "https://api.openai.com/v1/responses")
api_failures = [
    (APIConnectionError(request=request), "internet connection"),
    (APITimeoutError(request=request), "timeout"),
    (RateLimitError("Rate limit reached", response=httpx.Response(429, request=request), body=None), "quota"),
    (AuthenticationError("Invalid credentials", response=httpx.Response(401, request=request), body=None), "OPENAI_API_KEY"),
    (InternalServerError("Service unavailable", response=httpx.Response(500, request=request), body=None), "status code"),
]
for failure, suggestion in api_failures:
    def fail_create(**kwargs):
        raise failure
    failing_client = SimpleNamespace(responses=SimpleNamespace(create=fail_create))
    expect_workflow_error(lambda: call_listing_api(failing_client, []),
                          "call_listing_api", type(failure).__name__, suggestion)

from pydantic import BaseModel
class ValidationExample(BaseModel):
    quantity: int

@report_errors("Check the input schema.")
def validate_example():
    """Exercise reporting of a Pydantic validation failure."""
    return ValidationExample(quantity="invalid")

expect_workflow_error(validate_example, "validate_example", "ValidationError", "quantity")
batch_results, batch_errors = process_products([{}, sample_product], fake_client, n=2, delay=0)
assert len(batch_results) == 1 and len(batch_errors) == 1
assert "Function: validate_product" in batch_errors[0]["error"]
print("Failure checks passed: files, permissions, JSON, validation, images, response shape, network, timeout, rate limits, authentication, server errors, and batch continuation.")
print("Example diagnostic:")
print(example_error)

Failure checks passed: files, permissions, JSON, validation, images, response shape, network, timeout, rate limits, authentication, server errors, and batch continuation.
Example diagnostic:
Function: load_local_products | Error: FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\tscho\\AppData\\Local\\Temp\\tmpnsuk9qvb\\missing.json' | Context: path=WindowsPath('C:/Users/tscho/AppData/Local/Temp/tmpnsuk9qvb/missing.json'); source=good_practice.ipynb/cell-5:11 | Suggestion: Check the path and working directory; create missing parent directories before saving.
